# Session 2: Prompt Engineering for Agentic Behaviors (Instructor Notebook — Full Solutions)

## Learning Objectives

By the end of this session, students will be able to:

1. **Craft effective system prompts** that define agent personas, capabilities, and constraints
2. **Implement Chain-of-Thought (CoT) and ReAct prompting** patterns to improve reasoning quality
3. **Generate structured outputs** (JSON mode) for reliable downstream processing
4. **Manage multi-turn conversations** with context accumulation and summarization strategies

In [53]:
from dotenv import load_dotenv
import os
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# Imports and Setup
import openai
import json
import os

# Ensure your OPENAI_API_KEY is set in your environment variables
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

client = openai.OpenAI()
print("Setup complete. OpenAI client initialized.")

Setup complete. OpenAI client initialized.


---
## Demos

Walk through these demos with the students. Run each cell and discuss the outputs.

### Demo 1: Zero-Shot vs. Few-Shot Prompting

**Zero-shot** prompting asks the model to perform a task with no examples.  
**Few-shot** prompting provides a handful of examples so the model can learn the expected format and behavior.

We will compare both approaches on a **client feedback classification** task — categorizing engagement survey responses as positive, negative, or neutral.

**Instructor Note:** Emphasize that few-shot prompting is one of the simplest yet most effective techniques for controlling output format. Point out the difference in verbosity between the two responses.

In [54]:
# Demo 1: Zero-Shot vs. Few-Shot Prompting

# Zero-shot: no examples provided
zero_shot_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Classify the sentiment of this client engagement feedback as positive, negative, or neutral: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}]
)
print("Zero-shot:", zero_shot_response.choices[0].message.content)

print("\n" + "="*60 + "\n")

# Few-shot: provide labeled examples from past McKinsey engagement surveys
few_shot_messages = [
    {"role": "system", "content": "You are a McKinsey client feedback classifier. Respond with only: POSITIVE, NEGATIVE, or NEUTRAL."},
    {"role": "user", "content": "Feedback: 'The engagement exceeded our expectations — the team was exceptional and the insights transformed our strategy.'"},
    {"role": "assistant", "content": "POSITIVE"},
    {"role": "user", "content": "Feedback: 'Disappointed with the engagement. Deliverables were late and the analysis did not address our core issues.'"},
    {"role": "assistant", "content": "NEGATIVE"},
    {"role": "user", "content": "Feedback: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}
]
few_shot_response = client.chat.completions.create(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), messages=few_shot_messages)
print("Few-shot:", few_shot_response.choices[0].message.content)

Zero-shot: The sentiment of the feedback is negative. While the client acknowledges the solid analysis provided by the McKinsey team, the criticism regarding the lack of specificity in the final recommendations indicates dissatisfaction.


Few-shot: NEUTRAL


**Observation:** Notice how the zero-shot response may include extra explanation, while the few-shot response follows the exact format demonstrated in the examples.

### Demo 2: Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting encourages the model to show its reasoning process step by step, which improves accuracy on math, logic, and multi-step problems.

**Instructor Note:** The key insight here is that LLMs perform better when they "think out loud." This mirrors McKinsey's structured problem-solving — breaking a problem into steps before jumping to the answer. CoT is especially valuable for market sizing and financial analysis.

In [55]:
# Demo 2: Chain-of-Thought Prompting

problem = "A McKinsey client has 1,200 retail stores. They plan to close 20% of underperforming locations and increase revenue per remaining store by 15%. If current average revenue per store is $2M, what will total revenue be after the restructuring?"

# Without CoT
response_no_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": problem}]
)
print("Without CoT:")
print(response_no_cot.choices[0].message.content)

print("\n" + "="*60 + "\n")

# With explicit CoT — mirroring McKinsey structured problem-solving
cot_prompt = f"{problem}\n\nLet's solve this step by step:\n1. First, calculate how many stores will be closed\n2. Then, find the number of remaining stores\n3. Calculate the new revenue per store after the 15% increase\n4. Compute total projected revenue"
response_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": cot_prompt}]
)
print("With CoT:")
print(response_cot.choices[0].message.content)

Without CoT:
To calculate the total revenue after the proposed restructuring, we can follow these steps:

1. **Calculate the number of stores that will remain after closing 20% of the underperforming locations**:
   \[
   \text{Number of stores to close} = 20\% \text{ of } 1,200 = 0.20 \times 1,200 = 240
   \]
   \[
   \text{Remaining stores} = 1,200 - 240 = 960
   \]

2. **Calculate the new average revenue per remaining store after increasing it by 15%**:
   \[
   \text{Increase in revenue per store} = 15\% \text{ of } \$2,000,000 = 0.15 \times 2,000,000 = 300,000
   \]
   \[
   \text{New average revenue per store} = \$2,000,000 + \$300,000 = \$2,300,000
   \]

3. **Calculate the total revenue after the restructuring**:
   \[
   \text{Total revenue} = \text{Remaining stores} \times \text{New average revenue per store}
   \]
   \[
   \text{Total revenue} = 960 \times 2,300,000 = \$2,208,000,000
   \]

Therefore, the total revenue after the restructuring will be **$2.208 billion**.


Wi

**Observation:** The CoT version explicitly walks through each calculation, making it easier to verify correctness and debug any errors in reasoning.

### Demo 3: Role-Based Prompting for Agentic Personas

By changing the system prompt, we can create entirely different McKinsey practice-area personas that approach the same client question from different angles. This is foundational for building specialized agents in multi-agent consulting systems.

**Instructor Note:** Discuss how multi-agent systems leverage this — each agent can be a different practice area expert working on the same client problem, mirroring how McKinsey staffs cross-functional teams.

In [56]:
# Demo 3: Role-Based Prompting for Agentic Personas

personas = {
    "McKinsey Growth Strategy Lead": "You are a McKinsey partner in Growth & Innovation. You approach every question by identifying growth levers, market opportunities, and revenue acceleration strategies. Use frameworks like the Three Horizons of Growth.",
    "McKinsey Risk & Compliance Expert": "You are a McKinsey senior expert in Risk & Resilience. You evaluate everything through the lens of regulatory risk, compliance requirements, operational resilience, and enterprise risk management.",
    "McKinsey Organization & People Lead": "You are a McKinsey partner in People & Organizational Performance. You think about talent strategy, organizational design, change management, and leadership development. Focus on the human side of transformation."
}
question = "Our banking client is considering launching a digital-only subsidiary to compete with fintechs."

for role, system_prompt in personas.items():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=200
    )
    print(f"\n{'='*50}\n{role}:\n{response.choices[0].message.content}")


McKinsey Growth Strategy Lead:
Launching a digital-only subsidiary is a strategic move that can enhance your banking client's competitive position against fintechs. To support this initiative, we will adopt the Three Horizons of Growth framework, which helps in identifying growth levers, market opportunities, and revenue acceleration strategies. 

### Horizon 1: Defend and Optimize
Here, the objective is to enhance existing offerings while ensuring current customer retention.

#### Key Actions:
1. **Customer Insights and User Experience**:
   - Conduct surveys and focus groups to understand customer needs.
   - Design a seamless user experience focusing on easy navigation and quick transactions.

2. **Cost Optimization**:
   - Leverage technology to reduce operational costs associated with chasing fintech innovations (e.g., AI for customer service).

3. **Tiered Pricing Models**:
   - Introduce competitive pricing structures to attract various customer segments, including those unders

**Observation:** Each persona focuses on different aspects of the same question -- data metrics, security risks, or user/business impact. This demonstrates how system prompts shape agent behavior.

### Demo 4: Structured Output Generation (JSON Mode)

When building agentic systems for McKinsey workflows, we often need the LLM output to be machine-readable — for example, extracting structured client data from meeting notes or parsing engagement details from unstructured text. JSON mode ensures the response is valid JSON, making downstream processing reliable.

**Instructor Note:** Highlight that `response_format={"type": "json_object"}` is an API-level guarantee. Without it, the model might produce mostly-valid JSON but occasionally break format — unacceptable in production consulting tools.

In [57]:
# Demo 4: Structured Output Generation (JSON Mode)

text = "Sarah Chen is the CFO of Meridian Healthcare, a $3.2B revenue hospital network based in Chicago. She has 18 years of experience in healthcare finance and previously led M&A at Deloitte. Her priorities for the McKinsey engagement are cost optimization, revenue cycle management, and evaluating two potential acquisition targets."

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "Extract the client executive's information and return it as a JSON object with keys: name, title, company, company_revenue, location, experience_years, previous_employer, engagement_priorities (as a list)."},
        {"role": "user", "content": text}
    ],
    response_format={"type": "json_object"}
)

parsed = json.loads(response.choices[0].message.content)
print("Parsed JSON output:")
print(json.dumps(parsed, indent=2))

# Demonstrate that we can access individual fields programmatically
print(f"\nClient Executive: {parsed.get('name')}, {parsed.get('title')}")
print(f"Engagement Priorities: {', '.join(parsed.get('engagement_priorities', []))}")

Parsed JSON output:
{
  "name": "Sarah Chen",
  "title": "CFO",
  "company": "Meridian Healthcare",
  "company_revenue": "$3.2B",
  "location": "Chicago",
  "experience_years": 18,
  "previous_employer": "Deloitte",
  "engagement_priorities": [
    "cost optimization",
    "revenue cycle management",
    "evaluating two potential acquisition targets"
  ]
}

Client Executive: Sarah Chen, CFO
Engagement Priorities: cost optimization, revenue cycle management, evaluating two potential acquisition targets


**Observation:** With `response_format={"type": "json_object"}`, the output is guaranteed to be valid JSON, which we can reliably parse and use in code.

### Demo 5: Multi-Turn Conversation Management

Consulting agents need to maintain context across multiple exchanges — remembering the client industry, engagement scope, and prior recommendations. This demo shows how to build a conversation manager that accumulates context, mirroring how a McKinsey engagement progresses through multiple discussions.

**Instructor Note:** Discuss the trade-off between keeping full history (better context) and truncating/summarizing (staying within token limits). In long-running engagements with extensive documentation, you need a strategy for managing context window size.

In [58]:
# Demo 5: Multi-Turn Conversation Management

class ConversationManager:
    def __init__(self, system_prompt, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")):
        self.model = model
        self.messages = [{"role": "system", "content": system_prompt}]
        self.client = openai.OpenAI()
    
    def send(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            max_tokens=300
        )
        assistant_msg = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_msg})
        return assistant_msg
    
    def get_history_length(self):
        return len(self.messages)

# Demo: Multi-turn McKinsey engagement planning conversation
conv = ConversationManager("You are a McKinsey engagement planning assistant. Help consultants scope and plan client engagements. Be structured, use MECE principles, and provide actionable next steps.")

print("User: We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%.")
print("Assistant:", conv.send("We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%."))

print("\n" + "-"*50)
print("\nUser: What data should we request from the client in the first week?")
print("Assistant:", conv.send("What data should we request from the client in the first week?"))

print("\n" + "-"*50)
print("\nUser: Can you summarize our engagement scope so far?")
print("Assistant:", conv.send("Can you summarize our engagement scope so far?"))

print(f"\nConversation history: {conv.get_history_length()} messages")

User: We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%.
Assistant: Congratulations on the new engagement! To scope and plan the project effectively, we need to understand the problem and the context deeply while ensuring our plan is structured, MECE, and actionable. Below is a structured approach to plan this engagement.

### Engagement Scope

**Objective:**
Reduce claims processing costs by 30% for the insurance company.

**Key Questions:**
1. What are the current claims processing costs?
2. What are the key drivers contributing to these costs?
3. What processes are currently in place for claims processing?
4. What technology solutions are being utilized?
5. What best practices in claims processing can we benchmark against?
6. Are there regulatory or compliance factors to consider?

### MECE Breakdown of Engagement

1. **Baseline Assessment (Current State Analysis)**
   - Financial Analysis: Breakdown of current claims 

**Observation:** The assistant remembers context from earlier turns -- it knows we are discussing Japan without being reminded. The conversation history grows with each exchange.

### Demo 6: Structured Outputs with LangChain Prompt Templates

LangChain provides a higher-level abstraction for prompt engineering. Its `ChatPromptTemplate` supports parameterized prompts, and `OutputParser` classes enforce structured output formats (JSON, lists, etc.) — reducing boilerplate and improving reliability.

**Instructor Note:** Compare this with Demo 4 (raw JSON mode). LangChain templates are reusable, composable, and integrate with chains/agents. This is the industry-standard approach for production-grade prompt workflows.

In [59]:
# Demo 6: Structured Outputs with LangChain Prompt Templates

# Install if needed: pip install langchain langchain-openai
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# Step 1: Initialize the LangChain LLM wrapper
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)

# --- Part A: ChatPromptTemplate with variables ---
print("PART A: LangChain ChatPromptTemplate for McKinsey Analysis")
print("=" * 60)

# Reusable template for industry analysis across different client engagements
analysis_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey {practice_area} expert. Provide structured, partner-ready analysis."),
    ("user", "Analyze the following for our client engagement: {topic}\n\nProvide your analysis in exactly 3 bullet points with strategic implications.")
])

prompt_value = analysis_template.invoke({"practice_area": "Operations", "topic": "supply chain resilience in the semiconductor industry"})
response = llm.invoke(prompt_value)
print("Practice: Operations | Topic: semiconductor supply chain")
print(response.content)

print()
prompt_value2 = analysis_template.invoke({"practice_area": "Strategy & Corporate Finance", "topic": "consolidation trends in European banking"})
response2 = llm.invoke(prompt_value2)
print("Practice: Strategy | Topic: European banking consolidation")
print(response2.content)

# --- Part B: PydanticOutputParser for McKinsey deliverables ---
print("\n" + "=" * 60)
print("PART B: PydanticOutputParser \u2014 McKinsey Engagement Summary")
print("=" * 60)

# Define the output schema using Pydantic
class EngagementSummary(BaseModel):
    executive_summary: str = Field(description="A 1-2 sentence executive summary suitable for a CEO briefing")
    key_findings: List[str] = Field(description="A list of 3 key findings")
    strategic_priority: str = Field(description="The single most important strategic priority: growth, cost_optimization, digital_transformation, or M&A")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")

output_parser = PydanticOutputParser(pydantic_object=EngagementSummary)
format_instructions = output_parser.get_format_instructions()
print(f"Format instructions (auto-generated):\n{format_instructions}\n")

structured_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey engagement manager preparing a structured deliverable."),
    ("user", "Analyze the strategic landscape for: {topic}\n\n{format_instructions}")
])

prompt = structured_template.invoke({
    "topic": "a mid-size US retailer considering an omnichannel transformation",
    "format_instructions": format_instructions
})

response = llm.invoke(prompt)
parsed_output = output_parser.parse(response.content)

print("Parsed structured output:")
print(f"  executive_summary: {parsed_output.executive_summary}")
print(f"  key_findings: {parsed_output.key_findings}")
print(f"  strategic_priority: {parsed_output.strategic_priority}")
print(f"  confidence: {parsed_output.confidence}")

# --- Part C: LangChain Chain (Prompt | LLM | Parser) ---
print("\n" + "=" * 60)
print("PART C: LangChain Chain \u2014 Prompt | LLM | Parser")
print("=" * 60)

chain = structured_template | llm | output_parser

result = chain.invoke({
    "topic": "a healthcare provider evaluating AI-powered diagnostics for radiology",
    "format_instructions": format_instructions
})

print("Chain output (directly parsed):")
print(f"  executive_summary: {result.executive_summary}")
print(f"  key_findings: {result.key_findings}")
print(f"  strategic_priority: {result.strategic_priority}")
print(f"  confidence: {result.confidence}")

PART A: LangChain ChatPromptTemplate for McKinsey Analysis
Practice: Operations | Topic: semiconductor supply chain
1. **Supply Chain Vulnerabilities and Geopolitical Risks**: The semiconductor industry is highly susceptible to geopolitical tensions, particularly between major players like the U.S. and China. Recent disruptions, such as export controls and trade restrictions, have highlighted the fragility of global supply chains. **Strategic Implication**: Clients should diversify their supply sources and consider nearshoring or reshoring strategies to mitigate risks associated with geopolitical uncertainties and ensure a more resilient supply chain.

2. **Technological Advancements and Innovation**: The rapid pace of technological change in semiconductor manufacturing, including the shift towards advanced nodes and new materials, necessitates continuous investment in R&D and production capabilities. Companies that fail to innovate risk falling behind competitors. **Strategic Implicat

### Task 1: Design Effective System Prompts for a McKinsey Industry Research Agent

**Approach:** We design a detailed system prompt that defines the agent's role (McKinsey Industry Research Specialist), its analytical approach (MECE sub-questions, data-driven reasoning), the required output format (Executive Summary, Key Findings, Strategic Implications, Confidence Level, Data Gaps), and behavioral constraints (acknowledge uncertainty, flag assumptions). The prompt is 200+ words — longer system prompts yield more consistent, structured outputs.

In [60]:
# Task 1 Solution: Design Effective System Prompts for a McKinsey Research Agent

research_agent_prompt = """You are a McKinsey Industry Research Agent specializing in analyzing market dynamics, competitive landscapes, and strategic opportunities for client engagements.

## Your Approach:
1. Break down the research question into MECE sub-questions
2. Analyze each dimension with data-driven reasoning
3. Synthesize findings into a structured, partner-ready response

## Output Format:
Always structure your response as:
- **Executive Summary**: 2-3 sentence overview suitable for a CEO briefing
- **Key Findings**: Bullet points of main discoveries with quantitative evidence where possible
- **Strategic Implications**: What this means for the client's competitive position
- **Confidence Level**: High/Medium/Low with justification
- **Data Gaps**: What additional data would strengthen the analysis

## Guidelines:
- If uncertain, explicitly state your confidence level and flag assumptions
- Distinguish between facts, estimates, and inferences
- Use McKinsey-standard language and frameworks (Porter's Five Forces, value chain analysis, etc.)
- Provide balanced perspectives — always consider the 'so what?' for the client"""

def ask_research_agent(question):
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": research_agent_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=800
    )
    result = response.choices[0].message.content
    print(result)
    return result

ask_research_agent("What are the key growth drivers and competitive dynamics in the global electric vehicle battery market?")

**Executive Summary**: The global electric vehicle (EV) battery market is poised for substantial growth, driven by increasing demand for electric vehicles, significant advancements in battery technology, and supportive regulatory frameworks. However, competitive dynamics are intensifying, as established players and new entrants vie for market share amidst supply chain constraints and evolving consumer preferences.

**Key Findings**:
- **Market Growth Rate**: The global EV battery market is projected to grow at a compound annual growth rate (CAGR) of 20% through 2030, driven by a surge in EV adoption, which is expected to reach 50% of total vehicle sales by 2030.
- **Technological Innovations**: Lithium-ion batteries dominate the market, yet advancements in solid-state batteries and alternative chemistries (e.g., lithium iron phosphate) are emerging, potentially improving energy density and reducing costs.
- **Supply Chain Challenges**: The market faces raw material supply constraints, 

'**Executive Summary**: The global electric vehicle (EV) battery market is poised for substantial growth, driven by increasing demand for electric vehicles, significant advancements in battery technology, and supportive regulatory frameworks. However, competitive dynamics are intensifying, as established players and new entrants vie for market share amidst supply chain constraints and evolving consumer preferences.\n\n**Key Findings**:\n- **Market Growth Rate**: The global EV battery market is projected to grow at a compound annual growth rate (CAGR) of 20% through 2030, driven by a surge in EV adoption, which is expected to reach 50% of total vehicle sales by 2030.\n- **Technological Innovations**: Lithium-ion batteries dominate the market, yet advancements in solid-state batteries and alternative chemistries (e.g., lithium iron phosphate) are emerging, potentially improving energy density and reducing costs.\n- **Supply Chain Challenges**: The market faces raw material supply constra

### Task 2: Implement ReAct-Style Prompting (Reason + Act)

**Goal:** Implement the ReAct pattern where the model interleaves reasoning (Thought) with actions (Action) and observations (Observation) — applied to a McKinsey due diligence scenario.

**Key Teaching Points:**
- ReAct is a foundational pattern for agentic AI — it mirrors how McKinsey consultants structure their analysis: hypothesize, gather data, refine.
- In this demo, the LLM simulates both the reasoning and the tool results. In a real system, Actions would trigger actual data lookups.
- The explicit format makes the reasoning chain transparent and reviewable — critical for client-facing work.

In [61]:
# Task 2 Solution: Implement ReAct-Style Prompting (Reason + Act)

def react_agent(question):
    react_prompt = """You are a McKinsey due diligence ReAct agent that solves problems by interleaving reasoning and actions.

Available Actions:
- MarketData[query]: Look up market size, growth rates, and industry data
- FinancialAnalysis[expression]: Perform financial calculations (margins, multiples, CAGR)
- CompetitorLookup[company]: Research competitor positioning and market share

Follow this EXACT format for each step:
Thought: [Your hypothesis or reasoning about what data you need next]
Action: [ActionName][input]
Observation: [What the data shows]

Continue until you have enough information, then provide:
Final Answer: [Your complete, structured recommendation]

Always show your complete reasoning chain — this is a client-facing deliverable."""

    client = openai.OpenAI()
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": react_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=500
    )
    result = response.choices[0].message.content
    print(result)
    return result

react_agent("Is a $200M acquisition of a mid-size European logistics company a good deal for our private equity client, given the current market conditions?")

Thought: I need to assess the overall market size and growth prospects of the logistics sector in Europe to understand if the acquisition of a mid-size logistics company for $200M is justifiable. 
Action: MarketData[European logistics market size and growth rates]
Observation: The European logistics market is valued at approximately €300 billion with a projected annual growth rate (CAGR) of around 4.5% over the next five years. 

Thought: I need to analyze the financial performance and valuation metrics of similar logistics companies to evaluate if the $200M acquisition price aligns with industry norms. 
Action: FinancialAnalysis[average EBITDA multiples for logistics companies in Europe]
Observation: The average EBITDA multiple for mid-size logistics companies in Europe ranges between 8x to 12x, indicating healthy valuation benchmarks for acquisitions in this sector.

Thought: Next, I will look into competitor positioning and market share of the target company to understand its compet

"Thought: I need to assess the overall market size and growth prospects of the logistics sector in Europe to understand if the acquisition of a mid-size logistics company for $200M is justifiable. \nAction: MarketData[European logistics market size and growth rates]\nObservation: The European logistics market is valued at approximately €300 billion with a projected annual growth rate (CAGR) of around 4.5% over the next five years. \n\nThought: I need to analyze the financial performance and valuation metrics of similar logistics companies to evaluate if the $200M acquisition price aligns with industry norms. \nAction: FinancialAnalysis[average EBITDA multiples for logistics companies in Europe]\nObservation: The average EBITDA multiple for mid-size logistics companies in Europe ranges between 8x to 12x, indicating healthy valuation benchmarks for acquisitions in this sector.\n\nThought: Next, I will look into competitor positioning and market share of the target company to understand i

### Task 3: Create a Reusable Prompt Template Library

**Goal:** Build a `PromptTemplate` class that manages parameterized prompts with validation — tailored for common McKinsey deliverable types.

**Key Teaching Points:**
- Template libraries reduce duplication and enforce consistency across McKinsey engagement outputs.
- Validation prevents runtime errors from missing variables.
- This pattern is used in frameworks like LangChain (`PromptTemplate`) and is an industry best practice for scaling consulting AI tools.

In [62]:
# Task 3 Solution: Create a Reusable Prompt Template Library

import re

class PromptTemplate:
    def __init__(self, name, template, description=""):
        self.name = name
        self.template = template
        self.description = description
        self.variables = re.findall(r'\{(\w+)\}', template)
    
    def format(self, **kwargs):
        self.validate(**kwargs)
        return self.template.format(**kwargs)
    
    def validate(self, **kwargs):
        missing = [v for v in self.variables if v not in kwargs]
        if missing:
            raise ValueError(f"Missing required variables: {missing}")
        return True

# McKinsey consulting template library
market_sizing_template = PromptTemplate(
    name="market_sizing",
    template="Estimate the total addressable market (TAM) for {product_category} in {geography}. Break down using a {approach} approach and provide your assumptions in {length} bullet points.",
    description="Market sizing analysis for client engagements"
)

executive_briefing_template = PromptTemplate(
    name="executive_briefing",
    template="Prepare a {length}-sentence executive briefing on {topic} for a {industry} client CEO. Focus on strategic implications and actionable recommendations.",
    description="Executive briefing for C-suite client meetings"
)

competitive_analysis_template = PromptTemplate(
    name="competitive_analysis",
    template="Analyze the competitive landscape for {company} in the {industry} sector. Identify the top {num_competitors} competitors and evaluate them on: {evaluation_criteria}.",
    description="Competitive landscape analysis for strategy engagements"
)

# Demo usage
prompt = market_sizing_template.format(
    product_category="AI-powered diagnostic tools",
    geography="North America",
    approach="top-down",
    length="5"
)
print(f"Generated prompt:\n{prompt}")
print(f"\nTemplate variables: {market_sizing_template.variables}")

# Show validation error
print("\n--- Testing validation ---")
try:
    market_sizing_template.format(product_category="AI tools")
except ValueError as e:
    print(f"Validation error (expected): {e}")

Generated prompt:
Estimate the total addressable market (TAM) for AI-powered diagnostic tools in North America. Break down using a top-down approach and provide your assumptions in 5 bullet points.

Template variables: ['product_category', 'geography', 'approach', 'length']

--- Testing validation ---
Validation error (expected): Missing required variables: ['geography', 'approach', 'length']


### Task 4: Build a Multi-Step Reasoning Agent with Tool Descriptions

**Goal:** Build a complete agentic loop where the LLM acts as a McKinsey analyst, deciding which consulting tool to call, receiving simulated results, and iterating until it has a final recommendation.

**Key Teaching Points:**
- This is the core pattern of tool-using agents: perceive -> reason -> act -> observe -> repeat.
- JSON mode is essential so the agent's tool calls are machine-parseable.
- The `simulate_tool` method stands in for real integrations (market databases, financial models, CRM systems).
- The loop has a max-step limit to prevent infinite loops — an important safety measure for production systems.

In [63]:
# Task 4 Solution: Build a Multi-Step Reasoning Agent with Tool Descriptions

tools = [
    {"name": "market_database", "description": "Look up market size, growth rates, and industry trends", "parameters": "query (string)"},
    {"name": "financial_model", "description": "Run financial calculations: DCF, multiples, CAGR, margin analysis", "parameters": "expression (string)"},
    {"name": "benchmarking", "description": "Compare client KPIs against industry benchmarks", "parameters": "metric and industry (string)"}
]

class ToolAgent:
    def __init__(self):
        self.client = openai.OpenAI()
        tool_descriptions = "\n".join([f"- {t['name']}: {t['description']} (params: {t['parameters']})" for t in tools])
        self.system_prompt = f"""You are a McKinsey analyst agent with access to the following consulting tools:

{tool_descriptions}

When you need to use a tool, respond with JSON:
{{"thought": "your analytical reasoning", "tool": "tool_name", "input": "tool_input"}}

When you have enough information to provide a recommendation, respond with JSON:
{{"thought": "final synthesis", "tool": "final_answer", "input": "your complete recommendation"}}

Process one analytical step at a time. Think like a McKinsey consultant — be structured and hypothesis-driven."""
    
    def simulate_tool(self, tool_name, tool_input):
        simulated = {
            "market_database": f"Market data for '{tool_input}': Market size $45B, growing at 8.2% CAGR, top 3 players hold 55% share",
            "financial_model": f"Financial analysis for '{tool_input}': Implied EV/EBITDA multiple of 12.5x, IRR of 18.3% over 5-year hold period",
            "benchmarking": f"Benchmark for '{tool_input}': Client at 75th percentile vs peers, top quartile in revenue growth but below median in EBITDA margin"
        }
        return simulated.get(tool_name, "Tool not found")
    
    def process(self, question):
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": question}
        ]
        
        for step in range(5):  # Max 5 analytical steps
            response = self.client.chat.completions.create(
                model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
                messages=messages,
                response_format={"type": "json_object"},
                max_tokens=300
            )
            result = json.loads(response.choices[0].message.content)
            print(f"\nStep {step + 1}:")
            print(f"  Thought: {result.get('thought', 'N/A')}")
            print(f"  Tool: {result.get('tool', 'N/A')}")
            print(f"  Input: {result.get('input', 'N/A')}")
            
            if result.get("tool") == "final_answer":
                print(f"\nFinal Recommendation: {result['input']}")
                return result["input"]
            
            observation = self.simulate_tool(result["tool"], result["input"])
            print(f"  Observation: {observation}")
            messages.append({"role": "assistant", "content": response.choices[0].message.content})
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        
        return "Max steps reached"

agent = ToolAgent()
agent.process("Should our PE client acquire a specialty chemicals company with $80M revenue and 22% EBITDA margins? What's the market outlook and how does the target compare to peers?")


Step 1:
  Thought: To determine whether the PE client should acquire the specialty chemicals company, I need to look into the market outlook for the specialty chemicals industry to understand growth potential and trends. Additionally, I need to benchmark the target company's EBITDA margins against industry peers to evaluate its performance. I will start by gathering information about market size and growth rates.
  Tool: market_database
  Input: specialty chemicals market outlook, growth rates, trends
  Observation: Market data for 'specialty chemicals market outlook, growth rates, trends': Market size $45B, growing at 8.2% CAGR, top 3 players hold 55% share

Step 2:
  Thought: The specialty chemicals market has a significant size of $45B and is experiencing a healthy growth rate of 8.2% CAGR. This suggests a favorable market outlook, indicative of potential opportunities for the target company. Next, I will benchmark the target company's EBITDA margin of 22% against industry peers to

"The PE client should consider acquiring the specialty chemicals company given the healthy market outlook with an 8.2% CAGR and the company's strong EBITDA margin position at the 75th percentile. However, it should also be noted that while revenue growth is robust, there's potential for improving margins to align closer with the industry median. A thorough due diligence process should be implemented to identify areas for margin enhancement."

### Task 5: Build a Few-Shot Consulting Deliverable Classifier

**Approach:** We construct a few-shot prompt with 4 labeled examples covering different McKinsey deliverable types (strategy deck, due diligence report, market sizing, operational review). The system message constrains the model to respond with only the category name. We then test on 3 unseen descriptions to verify generalization. This reinforces Demo 1's few-shot pattern with a more complex, multi-class classification task.

In [64]:
# Task 5 - SOLUTION: Build a Few-Shot Consulting Deliverable Classifier

def classify_deliverable(description):
    """Classify a consulting deliverable description into a category using few-shot prompting.

    Categories: STRATEGY_DECK, DUE_DILIGENCE, MARKET_SIZING, OPERATIONAL_REVIEW
    """
    # Step 1: Define few-shot examples covering each category.
    few_shot_examples = [
        {"role": "user", "content": "A presentation analyzing competitive positioning and recommending three strategic growth initiatives for the CEO."},
        {"role": "assistant", "content": "STRATEGY_DECK"},
        {"role": "user", "content": "A report evaluating the target company's financial health, customer concentration risk, and integration synergies for a PE buyer."},
        {"role": "assistant", "content": "DUE_DILIGENCE"},
        {"role": "user", "content": "A bottom-up analysis estimating the total addressable market for plant-based proteins in Southeast Asia."},
        {"role": "assistant", "content": "MARKET_SIZING"},
        {"role": "user", "content": "A diagnostic identifying $50M in procurement savings through supplier consolidation and contract renegotiation."},
        {"role": "assistant", "content": "OPERATIONAL_REVIEW"},
    ]

    # Step 2: Build the message list with system + examples + new input.
    messages = [
        {"role": "system", "content": "You are a McKinsey deliverable classifier. Classify each description into exactly one category. Respond with ONLY the category name: STRATEGY_DECK, DUE_DILIGENCE, MARKET_SIZING, or OPERATIONAL_REVIEW."},
    ] + few_shot_examples + [
        {"role": "user", "content": description}
    ]

    # Step 3: Call the API.
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=messages,
        temperature=0,
        max_tokens=10
    )
    return response.choices[0].message.content.strip()


# Test with unseen deliverable descriptions
test_cases = [
    "A board presentation outlining three acquisition targets and a recommended M&A roadmap for the next 18 months.",
    "A warehouse network optimization study benchmarking fulfillment costs against industry leaders.",
    "An analysis calculating the serviceable addressable market for EV charging in urban Germany.",
]

print("Few-Shot Deliverable Classification Results:")
print("=" * 60)
for desc in test_cases:
    category = classify_deliverable(desc)
    print(f"\nInput: {desc[:80]}...")
    print(f"Category: {category}")

Few-Shot Deliverable Classification Results:

Input: A board presentation outlining three acquisition targets and a recommended M&A r...
Category: STRATEGY_DECK

Input: A warehouse network optimization study benchmarking fulfillment costs against in...
Category: OPERATIONAL_REVIEW

Input: An analysis calculating the serviceable addressable market for EV charging in ur...
Category: MARKET_SIZING


### Task 6: Chain-of-Thought Market Sizing Agent

**Approach:** We create a function that takes a market sizing question and uses a CoT system prompt to force the LLM to break the estimation into numbered assumptions and arithmetic steps. We compare the CoT output against a direct (non-CoT) answer to demonstrate the difference in transparency and verifiability — critical for McKinsey engagement work where partners need to audit every assumption.

In [65]:
# Task 6 - SOLUTION: Chain-of-Thought Market Sizing Agent

def market_sizing_cot(question):
    """Estimate a market size using chain-of-thought reasoning.

    Returns a dict with: direct_answer (no CoT) and cot_answer (with CoT).
    """
    # Step 1: Direct answer (no CoT).
    direct = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are a McKinsey market sizing expert. Give a concise estimate."},
            {"role": "user", "content": question}
        ],
        temperature=0,
        max_tokens=150
    ).choices[0].message.content

    # Step 2: CoT answer with explicit reasoning structure.
    cot_system = """You are a McKinsey market sizing expert. Break down every estimation using chain-of-thought reasoning.

Follow this exact structure:
1. CLARIFY: State what exactly you are estimating
2. ASSUMPTIONS: List each assumption with a specific number (e.g., population = 330M)
3. CALCULATION: Show each arithmetic step on its own line
4. ANSWER: State the final market size estimate
5. SENSITIVITY: Identify the 1-2 assumptions that most affect the result"""

    cot = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": cot_system},
            {"role": "user", "content": question}
        ],
        temperature=0,
        max_tokens=500
    ).choices[0].message.content

    return {"direct_answer": direct, "cot_answer": cot}


# Test with a McKinsey-style market sizing question
question = "What is the annual market size for premium dog food in the United States?"
result = market_sizing_cot(question)

print("DIRECT ANSWER (no CoT):")
print("=" * 60)
print(result["direct_answer"])

print("\nCHAIN-OF-THOUGHT ANSWER:")
print("=" * 60)
print(result["cot_answer"])

DIRECT ANSWER (no CoT):
As of 2023, the annual market size for premium dog food in the United States is estimated to be around $10 billion. This segment has been growing steadily due to increasing consumer awareness of pet health and nutrition, as well as a trend towards higher-quality ingredients.

CHAIN-OF-THOUGHT ANSWER:
1. **CLARIFY**: We are estimating the annual market size for premium dog food in the United States.

2. **ASSUMPTIONS**:
   - Total number of households in the U.S. = 130 million
   - Percentage of households that own dogs = 60%
   - Average number of dogs per dog-owning household = 1.5
   - Percentage of dog owners that purchase premium dog food = 30%
   - Average annual spending on premium dog food per dog = $500

3. **CALCULATION**:
   - Total households in the U.S. = 130 million
   - Households with dogs = 130 million * 60% = 78 million
   - Total number of dogs = 78 million * 1.5 = 117 million
   - Households purchasing premium dog food = 78 million * 30% = 23.

### Task 7: Extract Structured Data from Client Meeting Notes

**Approach:** We use JSON mode to extract multiple structured entities from a block of unstructured meeting notes. The system prompt defines the exact JSON schema expected. We then validate the output by checking required fields and printing a summary. This extends Demo 4 by handling a more complex, multi-entity extraction task — a common need when processing consulting engagement notes at scale.

In [66]:
# Task 7 - SOLUTION: Extract Structured Data from Client Meeting Notes

def extract_meeting_data(meeting_notes):
    """Extract structured data from unstructured meeting notes using JSON mode.

    Returns a parsed dict with: attendees, key_decisions, action_items, risks.
    """
    # Step 1: Define the extraction schema in the system prompt.
    system_prompt = """You are a McKinsey engagement coordinator. Extract structured data from meeting notes.

Return a JSON object with exactly these fields:
{
  "meeting_date": "YYYY-MM-DD or 'unknown'",
  "client_name": "string",
  "attendees": [{"name": "string", "role": "string", "organization": "McKinsey or client company"}],
  "key_decisions": ["string"],
  "action_items": [{"owner": "string", "task": "string", "deadline": "string or 'TBD'"}],
  "risks_raised": ["string"],
  "next_meeting": "string or 'TBD'"
}"""

    # Step 2: Call with JSON mode.
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Extract data from these meeting notes:\n\n{meeting_notes}"}
        ],
        temperature=0,
        max_tokens=1000,
        response_format={"type": "json_object"}
    )

    # Step 3: Parse and validate.
    data = json.loads(response.choices[0].message.content)
    return data


# Test with realistic meeting notes
notes = """Kickoff meeting for Project Phoenix with Apex Manufacturing.
Date: March 15, 2025. Held at client HQ in Detroit.

Attendees: Sarah Kim (McKinsey EM), David Park (McKinsey Associate),
Tom Richardson (Apex CEO), Lisa Zhang (Apex VP Operations), Mike Torres (Apex CFO).

Tom opened by stressing urgency — margins dropped 400bps last year. Lisa flagged
that two major suppliers increased pricing by 18%. Mike wants a clear ROI framework
before any capex decisions.

Decisions: (1) Focus the first sprint on procurement and supplier consolidation.
(2) Defer factory automation analysis to Phase 2.

Action items: Sarah to deliver preliminary spend analysis by March 22.
David to schedule interviews with top 5 suppliers by end of week.
Lisa to share last 3 years of purchase order data.

Risks: supplier relationship damage if consolidation is too aggressive.
Union concerns if headcount implications arise in Phase 2.

Next meeting: March 22 at 9am, McKinsey Detroit office."""

result = extract_meeting_data(notes)

print("Extracted Meeting Data:")
print("=" * 60)
print(f"Client: {result['client_name']}")
print(f"Date: {result['meeting_date']}")
print(f"\nAttendees ({len(result['attendees'])}):'")
for a in result['attendees']:
    print(f"  - {a['name']} ({a['role']}, {a['organization']})")
print(f"\nKey Decisions:")
for d in result['key_decisions']:
    print(f"  - {d}")
print(f"\nAction Items ({len(result['action_items'])}):'")
for item in result['action_items']:
    print(f"  - [{item['owner']}] {item['task']} (by {item['deadline']})")
print(f"\nRisks: {result['risks_raised']}")
print(f"Next Meeting: {result['next_meeting']}")

Extracted Meeting Data:
Client: Apex Manufacturing
Date: 2025-03-15

Attendees (5):'
  - Sarah Kim (Engagement Manager, McKinsey)
  - David Park (Associate, McKinsey)
  - Tom Richardson (CEO, Apex)
  - Lisa Zhang (VP Operations, Apex)
  - Mike Torres (CFO, Apex)

Key Decisions:
  - Focus the first sprint on procurement and supplier consolidation.
  - Defer factory automation analysis to Phase 2.

Action Items (3):'
  - [Sarah Kim] Deliver preliminary spend analysis (by 2025-03-22)
  - [David Park] Schedule interviews with top 5 suppliers (by end of week)
  - [Lisa Zhang] Share last 3 years of purchase order data (by TBD)

Risks: ['Supplier relationship damage if consolidation is too aggressive.', 'Union concerns if headcount implications arise in Phase 2.']
Next Meeting: 2025-03-22 09:00


### Task 8: Build a Conversation Summarizer for Long Engagements

**Approach:** We extend the `ConversationManager` from Demo 5 with an auto-summarization feature. When the message history exceeds a configurable threshold, the agent uses an LLM call to compress the older messages into a single summary message, then continues the conversation with that summary plus the recent messages. This is a critical pattern for production agents that need to maintain coherent long-running conversations within context window limits.

In [67]:
# Task 8 - SOLUTION: Build a Conversation Summarizer for Long Engagements

class SmartConversationManager:
    def __init__(self, system_prompt, model=None, max_history=10):
        """Initialize with a system prompt and max message history before summarization."""
        self.client = openai.OpenAI()
        self.model = model or os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")
        self.system_prompt = system_prompt
        self.max_history = max_history
        self.messages = [{"role": "system", "content": system_prompt}]
        self.summaries = []  # Track summaries for debugging

    def _summarize_history(self):
        """Compress older messages into a summary, keeping the most recent exchanges."""
        # Step 1: Separate system message from conversation.
        conversation = self.messages[1:]  # everything except system

        # Step 2: Format conversation for summarization.
        conv_text = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in conversation])

        # Step 3: Ask LLM to summarize.
        summary_response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": "Summarize this conversation concisely. Preserve key facts: client name, industry, engagement scope, decisions made, and open questions. Use bullet points."},
                {"role": "user", "content": f"Summarize this engagement conversation:\n\n{conv_text}"}
            ],
            temperature=0,
            max_tokens=200
        ).choices[0].message.content

        self.summaries.append(summary_response)

        # Step 4: Replace history with system + summary + last 2 exchanges.
        recent = conversation[-4:]  # Keep last 2 user/assistant pairs
        self.messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "system", "content": f"[CONVERSATION SUMMARY]\n{summary_response}"},
        ] + recent

        print(f"  [Auto-summarized {len(conversation)} messages -> {len(self.messages)} messages]")

    def send(self, user_message):
        """Send a message, auto-summarizing if history exceeds threshold."""
        # Step 1: Check if we need to summarize.
        if len(self.messages) > self.max_history:
            self._summarize_history()

        # Step 2: Add user message and get response.
        self.messages.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            temperature=0.5,
            max_tokens=300
        )
        reply = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": reply})
        return reply

    def get_stats(self):
        """Return conversation statistics."""
        return {
            "current_messages": len(self.messages),
            "summaries_created": len(self.summaries),
        }


# Test with a multi-turn McKinsey engagement conversation
agent = SmartConversationManager(
    system_prompt="You are a McKinsey engagement planning assistant. Help consultants plan and track client engagements. Use MECE frameworks and be concise.",
    max_history=8  # Low threshold to trigger summarization quickly for demo
)

turns = [
    "We just won an engagement with a mid-size telecom company in Brazil. They want to reduce churn.",
    "What data should we request in the first week?",
    "The client shared that churn is 3.2% monthly, mostly prepaid customers.",
    "What hypotheses should we test about root causes?",
    "We found that 60% of churners cite network quality. The other 40% cite pricing.",
    "Given these findings, what should our final recommendation deck look like?",
]

for i, msg in enumerate(turns, 1):
    print(f"\n--- Turn {i} ---")
    print(f"USER: {msg}")
    reply = agent.send(msg)
    print(f"AGENT: {reply[:200]}...")

stats = agent.get_stats()
print(f"\nConversation stats: {stats}")


--- Turn 1 ---
USER: We just won an engagement with a mid-size telecom company in Brazil. They want to reduce churn.
AGENT: Great news on winning the engagement! To effectively plan and track the client engagement focused on reducing churn for the mid-size telecom company in Brazil, we can use a MECE framework. Here’s a st...

--- Turn 2 ---
USER: What data should we request in the first week?
AGENT: In the first week, it's crucial to gather foundational data that will inform your understanding of the current churn situation and customer dynamics. Here’s a structured list of data to request:

### ...

--- Turn 3 ---
USER: The client shared that churn is 3.2% monthly, mostly prepaid customers.
AGENT: Thank you for the additional information. Given that the churn rate is 3.2% monthly and primarily involves prepaid customers, we can refine our data request and analysis approach. Here’s how to procee...

--- Turn 4 ---
USER: What hypotheses should we test about root causes?
AGENT: To effe

---
## Summary

### Key Takeaways

1. **Zero-shot vs. Few-shot**: Few-shot prompting provides examples that guide the model toward consistent format and behavior. Use it when you need predictable, structured responses.

2. **Chain-of-Thought (CoT)**: Explicitly asking for step-by-step reasoning improves accuracy on complex tasks. CoT is especially valuable for math, logic, and multi-step problems.

3. **Role-Based Prompting**: System prompts are the primary mechanism for defining agent personas. A well-crafted system prompt shapes the entire behavior profile of an agent.

4. **Structured Outputs**: JSON mode ensures machine-readable responses, which is critical for agentic workflows where outputs feed into downstream code or other agents.

5. **Multi-Turn Management**: Maintaining conversation history enables context-aware interactions, but requires careful management of the context window as conversations grow.

6. **LangChain Prompt Templates**: LangChain's `ChatPromptTemplate` and `PydanticOutputParser` provide reusable, composable abstractions for prompt engineering. The pipe operator (`|`) lets you chain prompts, LLMs, and parsers into end-to-end workflows.

### Looking Ahead

These prompt engineering techniques form the foundation for building agentic systems. In the next session, we will explore how to evaluate and measure the quality of agent outputs systematically.